# 第41篇｜多元线性回归：多个自变量同时建模

> 这是「数据分析从入门到精通」系列的第 41 篇。一个自变量的情况搞定了，现实中影响结果的往往不止一个因素。这篇来聊多元线性回归——多个自变量同时建模，同时学会怎么解读每个系数的含义。

---

嗨，我是小荷～

一元回归只能用一个变量预测，但现实中影响结果的因素往往不止一个。房价取决于面积、地段、楼层、建造年份……这时候就需要**多元线性回归**。

---

## 一、多元线性回归模型

来用代码实现线性回归：


In [ ]:
y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ + ε


每个 βᵢ 的含义：**在其他变量保持不变的情况下，xᵢ 增加 1 个单位，y 变化的量**（偏回归系数）。

---

## 二、完整建模示例

多元回归跟一元回归流程一样，只是多了几个自变量：


In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']   # 文泉驿微米黑
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(2024)
n = 500

df = pd.DataFrame({
    '面积':      np.random.uniform(50, 200, n),
    '距地铁km':  np.random.uniform(0.1, 5.0, n),
    '建造年份':  np.random.randint(1990, 2024, n),
    '楼层':      np.random.randint(1, 30, n),
    '学区':      np.random.choice([0, 1], n, p=[0.6, 0.4])   # 是否学区
})

df['房价万元'] = (
    df['面积'] * 3.5
    - df['距地铁km'] * 15
    + (2024 - df['建造年份']) * (-0.8)
    + df['楼层'] * 0.5
    + df['学区'] * 80
    + 50
    + np.random.normal(0, 30, n)
)

X = df.drop('房价万元', axis=1)
y = df['房价万元']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# 模型评估
y_pred = model.predict(X_test)
print("多元线性回归结果：")
print(f"测试集 R²  = {r2_score(y_test, y_pred):.4f}")
print(f"测试集 MAE = {mean_absolute_error(y_test, y_pred):.1f} 万元")
print()
print("各特征系数：")
for feat, coef in zip(X.columns, model.coef_):
    print(f"  {feat:12s}: {coef:.3f}")
print(f"  {'截距':12s}: {model.intercept_:.3f}")


多元线性回归结果：
测试集 R²  = 0.9550
测试集 MAE = 28.6 万元

各特征系数：
  面积          : 3.498
  距地铁km       : -16.536
  建造年份        : 0.746
  楼层          : 0.045
  学区          : 79.067
  截距          : -1451.838


---

## 三、statsmodels 完整统计报告

多元回归要看的东西更多——每个系数的显著性、整体 F 检验、调整 R²……：


In [4]:
X_sm = sm.add_constant(X_train)
model_sm = sm.OLS(y_train, X_sm).fit()
print(model_sm.summary())


                            OLS Regression Results                            
Dep. Variable:                   房价万元   R-squared:                       0.962
Model:                            OLS   Adj. R-squared:                  0.961
Method:                 Least Squares   F-statistic:                     1991.
Date:                Mon, 25 May 2026   Prob (F-statistic):          4.69e-277
Time:                        16:46:05   Log-Likelihood:                -1929.2
No. Observations:                 400   AIC:                             3870.
Df Residuals:                     394   BIC:                             3894.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1451.8384    310.036     -4.683      0.0

关注：
- **coef**：每个变量的回归系数
- **P>|t|**：系数是否显著（< 0.05 表示该变量对 y 有显著影响）
- **R-squared**：模型整体解释力
- **F-statistic & Prob(F)**：整体模型是否显著

---

## 四、标准化系数（比较变量重要性）

原始系数不能直接比较（量纲不同），标准化后可以：


In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

model_std = LinearRegression()
model_std.fit(X_scaled, y_train)

print("\n标准化系数（绝对值越大 = 影响越大）：")
coef_sorted = sorted(zip(X.columns, model_std.coef_), key=lambda x: abs(x[1]), reverse=True)
for feat, coef in coef_sorted:
    bar = '█' * int(abs(coef) * 10)
    print(f"  {feat:12s}: {coef:+.3f}  {bar}")



标准化系数（绝对值越大 = 影响越大）：
  面积          : +145.995  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

---

## 五、共线性初探

如果多个自变量之间高度相关（多重共线性），回归系数会变得不稳定：


In [6]:
# 相关矩阵检查多重共线性
corr = X.corr()
print("\n特征间相关矩阵：")
print(corr.round(3))

# 如果任何两个特征的 |r| > 0.8，要注意共线性问题
high_corr = [(c1, c2, corr[c1][c2]) for c1 in corr.columns for c2 in corr.columns
              if c1 < c2 and abs(corr[c1][c2]) > 0.7]
if high_corr:
    print("\n警告：以下特征存在高度相关：")
    for c1, c2, r in high_corr:
        print(f"  {c1} & {c2}: r = {r:.3f}")
else:
    print("\n✅ 未发现严重共线性问题")



特征间相关矩阵：
          面积  距地铁km   建造年份     楼层     学区
面积     1.000  0.028 -0.053 -0.049  0.015
距地铁km  0.028  1.000 -0.027 -0.030 -0.028
建造年份  -0.053 -0.027  1.000  0.023 -0.027
楼层    -0.049 -0.030  0.023  1.000  0.030
学区     0.015 -0.028 -0.027  0.030  1.000

✅ 未发现严重共线性问题


---

## 六、预测新样本

模型建好了，拿新数据来预测一下试试：


In [7]:
# 预测一套新房子的价格
new_house = pd.DataFrame({
    '面积': [120],
    '距地铁km': [0.8],
    '建造年份': [2015],
    '楼层': [15],
    '学区': [1]
})

pred_price = model.predict(new_house)[0]
print(f"\n预测房价：{pred_price:.1f} 万元")



预测房价：538.0 万元


---

## 七、📝 小结

| 概念 | 含义 |
|------|------|
| 偏回归系数 | 控制其他变量，某变量单独对 y 的影响 |
| 标准化系数 | 消除量纲，比较变量相对重要性 |
| 多重共线性 | 自变量间高度相关，导致系数不稳定 |
| Adjusted R² | 调整后 R²，考虑了变量数量 |

---

## 八、🏋️ 课后练习

1. 利用下面的模拟数据。加入新特征"停车位数量"，看 R² 是否提升，系数是否显著。
2. 比较标准化前后各特征的"相对重要性"排名，是否一致？
3. 用模型预测"面积 80㎡、距地铁 0.5km、2020 年建造、5 楼、非学区"的房价。

In [8]:
# 模拟数据
np.random.seed(42)
n = 200

# 生成特征数据
area = np.random.uniform(50, 200, n)                    # 面积: 50-200平米
distance_subway = np.random.uniform(0.1, 5, n)          # 距地铁: 0.1-5km
year_built = np.random.randint(2000, 2025, n)           # 建造年份: 2000-2024
floor = np.random.randint(1, 33, n)                    # 楼层: 1-32
school_district = np.random.randint(0, 2, n)           # 是否学区: 0或1
parking = np.random.randint(0, 3, n)                    # 停车位数量: 0,1,2

# 生成房价（加入一些真实关系）
# 房价 = 0.5*面积 + (-5)*距地铁 + 0.3*年份 + 噪声
price = (0.5 * area
         - 5 * distance_subway
         + 0.3 * (year_built - 2000)
         + 2 * floor
         + 10 * school_district
         + 3 * parking
         + np.random.normal(0, 10, n)) * 10000  # 单位：元

# 创建DataFrame
df = pd.DataFrame({
    '面积(㎡)': area,
    '距地铁(km)': distance_subway,
    '建造年份': year_built,
    '楼层': floor,
    '是否学区': school_district,
    '停车位': parking,
    '房价(万元)': price / 10000
})

print("\n房价数据集预览：")
print(df.head(10).to_string())
print(f"\n数据形状: {df.shape}")


房价数据集预览：
        面积(㎡)   距地铁(km)  建造年份  楼层  是否学区  停车位      房价(万元)
0  106.181018  3.245955  2008  21     0    1  108.559211
1  192.607146  0.512286  2016  23     1    1  177.039470
2  159.799091  0.891981  2016   5     0    0   84.601978
3  139.798773  4.502916  2019  21     1    0   69.402756
4   73.402796  3.071502  2015   5     1    1   45.278565
5   73.399178  0.145066  2024  10     1    0   75.557224
6   58.712542  0.597211  2021  10     0    1   49.993028
7  179.926422  3.351159  2012  19     0    2  118.592197
8  140.167252  0.124802  2018  17     0    1  103.573037
9  156.210887  0.887959  2016  21     1    2  145.681993

数据形状: (200, 7)


In [9]:
# 练习1：加入新特征"停车位数量"，看R²是否提升
print("练习1：加入停车位特征前后的R²对比")

# 原始特征（不含停车位）
features_no_parking = ['面积(㎡)', '距地铁(km)', '建造年份', '楼层', '是否学区']
X_no_parking = df[features_no_parking]
X_with_parking = df[features_no_parking + ['停车位']]
y = df['房价(万元)']

# 模型1：不含停车位
model1 = LinearRegression()
model1.fit(X_no_parking, y)
r2_no_parking = model1.score(X_no_parking, y)

# 模型2：含停车位
model2 = LinearRegression()
model2.fit(X_with_parking, y)
r2_with_parking = model2.score(X_with_parking, y)

print(f"\n模型1（不含停车位）R²: {r2_no_parking:.4f}")
print(f"模型2（含停车位）R²: {r2_with_parking:.4f}")
print(f"R² 提升: {(r2_with_parking - r2_no_parking):.4f}")

# 显示各特征的系数
print("\n模型2各特征系数：")
print("-" * 40)
for feature, coef in zip(features_no_parking + ['停车位'], model2.coef_):
    print(f"  {feature}: {coef:.4f}")

# 系数是否显著判断（绝对值大小）
print("\n系数重要性排名（按绝对值）：")
coef_importance = sorted(zip(features_no_parking + ['停车位'], np.abs(model2.coef_)),
                        key=lambda x: x[1], reverse=True)
for rank, (feature, importance) in enumerate(coef_importance, 1):
    print(f"  {rank}. {feature}: {importance:.4f}")

练习1：加入停车位特征前后的R²对比

模型1（不含停车位）R²: 0.9008
模型2（含停车位）R²: 0.9048
R² 提升: 0.0040

模型2各特征系数：
----------------------------------------
  面积(㎡): 0.5129
  距地铁(km): -5.1962
  建造年份: 0.2256
  楼层: 2.0565
  是否学区: 9.3318
  停车位: 2.5430

系数重要性排名（按绝对值）：
  1. 是否学区: 9.3318
  2. 距地铁(km): 5.1962
  3. 停车位: 2.5430
  4. 楼层: 2.0565
  5. 面积(㎡): 0.5129
  6. 建造年份: 0.2256


In [10]:
print("练习2：标准化前后的特征重要性排名")

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_with_parking)

# 标准化后建模
model_scaled = LinearRegression()
model_scaled.fit(X_scaled, y)

print("\n标准化前系数（原始尺度）：")
for feature, coef in zip(features_no_parking + ['停车位'], model2.coef_):
    print(f"  {feature}: {coef:.4f}")

print("\n标准化后系数（标准化尺度）：")
coef_scaled = model_scaled.coef_
for feature, coef in zip(features_no_parking + ['停车位'], coef_scaled):
    print(f"  {feature}: {coef:.4f}")

# 排名对比
print("\n重要性排名对比：")
print("-" * 50)
print(f"{'特征':<12} {'原始系数排名':<15} {'标准化系数排名':<15}")
print("-" * 50)

# 原始系数排名
raw_rank = sorted(zip(features_no_parking + ['停车位'], model2.coef_),
                 key=lambda x: np.abs(x[1]), reverse=True)
# 标准化系数排名
scaled_rank = sorted(zip(features_no_parking + ['停车位'], coef_scaled),
                     key=lambda x: np.abs(x[1]), reverse=True)

for i in range(len(raw_rank)):
    raw_feat, raw_coef = raw_rank[i]
    scaled_feat, scaled_coef = scaled_rank[i]
    match = "✓" if raw_feat == scaled_feat else "×"
    print(f"{raw_feat:<12} {i+1}. {raw_feat:<10} {i+1}. {scaled_feat:<10} {match}")

练习2：标准化前后的特征重要性排名

标准化前系数（原始尺度）：
  面积(㎡): 0.5129
  距地铁(km): -5.1962
  建造年份: 0.2256
  楼层: 2.0565
  是否学区: 9.3318
  停车位: 2.5430

标准化后系数（标准化尺度）：
  面积(㎡): 22.6326
  距地铁(km): -7.4415
  建造年份: 1.5893
  楼层: 18.7596
  是否学区: 4.6657
  停车位: 2.1200

重要性排名对比：
--------------------------------------------------
特征           原始系数排名          标准化系数排名        
--------------------------------------------------
是否学区         1. 是否学区       1. 面积(㎡)      ×
距地铁(km)      2. 距地铁(km)    2. 楼层         ×
停车位          3. 停车位        3. 距地铁(km)    ×
楼层           4. 楼层         4. 是否学区       ×
面积(㎡)        5. 面积(㎡)      5. 停车位        ×
建造年份         6. 建造年份       6. 建造年份       ✓


In [11]:
print("练习3：预测指定条件的房价")

# 预测条件：面积80㎡、距地铁0.5km、2020年建造、5楼、非学区
new_house = pd.DataFrame({
    '面积(㎡)': [80],
    '距地铁(km)': [0.5],
    '建造年份': [2020],
    '楼层': [5],
    '是否学区': [0],
    '停车位': [1]  # 假设有1个停车位
})

predicted_price = model2.predict(new_house)[0]

print("\n预测条件：")
print(f"  面积: 80㎡")
print(f"  距地铁: 0.5km")
print(f"  建造年份: 2020年")
print(f"  楼层: 5楼")
print(f"  是否学区: 否")
print(f"  停车位: 1个")
print(f"\n预测房价: {predicted_price:.2f} 万元")

练习3：预测指定条件的房价

预测条件：
  面积: 80㎡
  距地铁: 0.5km
  建造年份: 2020年
  楼层: 5楼
  是否学区: 否
  停车位: 1个

预测房价: 55.79 万元


本篇完整代码包括练习题解答都已经上传至 GitHub 仓库，欢迎 Clone。

---

## 下期预告

> **第 42 篇：回归模型评估 — R² / MSE / 残差分析**
>
> 有了多元回归，下篇来学如何评估模型好坏：R²、MSE、残差分析，判断「我的模型到底好不好」。

---

*跟着小荷，数据分析路上不迷路～*